In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
# Load the dataset
path = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# print(df["Delivery_Time"].value_counts())
# print(df["Delivery_Time"].value_counts(normalize=True))
# Is the target Imbalanced ?

def check_imbalance(df, target):
    print("Target Dist")
    print(df[target].value_counts())

    plt.hist(x=df[target], bins=30)
    plt.show()



# Distribution
check_imbalance(df,"Delivery_Time" )


In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1)
df

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")
check_missing_values(df)

# Handling

# f['col'].fillna(0) # or 'unknown' for categorical
# Fill with statistics
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean()) # Mean (for numerical)
df['Delivery_Time'].fillna(df['Delivery_Time'].median()) # Median (robust to outliers)
df['Weather'].fillna(df['Weather'].mode()[0]) # Mode (for categorical/numerical)



In [ ]:
df['Courier_Experience_yrs'] =df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean()) # Mean (for numerical)
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].median()) # Median (robust to outliers)
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0]) # Mode (for categorical/numerical)
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0]) # Mode (for categorical/numerical)
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0]) # Mode (for categorical/numerical)


In [ ]:
check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# One-Hot Encoding (for nominal categories in linear models)
# Do We have categorical cols ?
categorical_cols = df.select_dtypes(include=["object"]).columns

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

# print("Categorical Columns:", list(categorical_cols))
# onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# for col in categorical_cols:
#   df[col] = onehot_encoder.fit_transform(df[[col]])
# df


In [ ]:
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
def check_imbalance(df, target):
    print("Target Dist")
    print(df[target].value_counts())

    plt.hist(x=df[target], bins=30)
    plt.show()

check_imbalance(df,"Delivery_Time" )

In [ ]:
# Taks
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
# from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score



models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': [], 'mae': []}


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


kf = KFold(n_splits=5, shuffle=True, random_state=42)

n_splits = 6
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)
    all_results[model_name]["mae"].append(mae)
    all

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  # print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  # print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  # print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")
  print(f"  mae:    {np.mean(all_results[model_name]['mae']):.4f}")

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Task 1: Write your code here:
# Retrieve CatBoost feature importances and sort them
forest_regressor = models["Random Forest Regressor"]
forest_regressor= list(zip(X.columns, forest_regressor.feature_importances_))
sorted_forest_regressor_importance = sorted(forest_regressor, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_forest_regressor_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
%pip install catboost

In [ ]:
# Define classification models
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost Regressor": CatBoostRegressor(),
}


models["Ensemble"] = VotingRegressor(estimators=[
        ('lr', models["CatBoost Regressor"]), ('gnb', models["Random Forest Regressor"])])

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(mean_absolute_error(y_Test, y_pred))

    # Print the results
    print(f"{model_name} mean absolute error: {np.mean(scores_f1):.4f}")
    print("\n")
